In [ ]:
import numpy as np, matplotlib.pyplot as plt
from pyhnc import *

In [ ]:
N = 2**13
L = 250
grid = Grid(L, N)
r, q = grid.r, grid.q

verbose = False

alpha = 0.5
niters = 1000
tol = 1e-12
solvent = Solver(grid, alpha=alpha, niters=niters, tol=tol)
solvent_rpa = RandomPhaseApproximationSolver(grid, alpha=alpha, niters=niters, tol=tol)

print(grid)
print(solvent)

In [ ]:
# Parameters for solvent-solvent interactions
A00 = 25
ρ0 = 3.0
φ0 = potentials.DPD(A00)

solvent = solvent.solve(φ0, ρ0, monitor=verbose)
solvent_rpa = solvent_rpa.solve(φ0, ρ0, monitor=verbose)

plt.figure(figsize=(3.375, 3))
plt.plot(r, solvent.g, label='HNC')
plt.plot(r, solvent_rpa.g, label='RPA')
plt.legend(loc='best')
plt.xlabel('$r$')
plt.ylabel('$g(r)$')
plt.xlim([0, 3])
plt.ylim([0, 1.5])
plt.show()

In [ ]:
solute = SoluteSolver(solvent)
solute_rpa = SoluteTestParticleRPA(solvent_rpa)

In [ ]:
A01 = 10
A02 = 2*A01
φ01 = potentials.DPD(A01)
φ02 = potentials.DPD(A02)

sol1 = solute.solve(φ01, monitor=verbose)
sol2 = solute.solve(φ02, monitor=verbose)
sol2_rpa = solute_rpa.solve(φ02, monitor=verbose)

ψ01 = grid.fourier_bessel_backward(sol1.hq * solvent.Sq**0.5)
ψ01q = grid.fourier_bessel_forward(ψ01)
ψ01_squ_q = grid.fourier_bessel_forward(ψ01**2)
h02q_add = (2*ψ01q - ψ01_squ_q) / solvent.Sq**0.5
g02_add = 1 + grid.fourier_bessel_backward(h02q_add)

plt.figure(figsize=(3.375, 3.375))
plt.plot(r, solvent.g, lw=0.5, label=f'$A_{{00}} = {A00}$')
plt.plot(r, sol1.g, lw=0.5, label=f'$A_{{01}} = {A01}$')
pl, = plt.plot(r, sol2.g, lw=0.5, label=f'$A_{{02}} = 2A_{{01}} = {A02}$')
plt.plot(r, g02_add, ':', lw=0.5, c=pl.get_color(),
         label=r'$\psi_2 = 2 \psi_1 - \psi_1^2$')
plt.plot(r, 1 + (2*sol1.h - sol1.h**2), '--', lw=0.5, c=pl.get_color(),
         label=r'$h_{02} = 2 h_{01} - h_{01}^2$')
# plt.plot(r, sol2_rpa.g, '-.', lw=0.5, c=pl.get_color(), label=r'TP-RPA')

plt.axhline(y=0)
plt.legend(loc='best')
plt.xlim([0, 2])
plt.ylim([0, 1.3])
plt.xlabel('$r$')
plt.ylabel('$g(r)$')
plt.show()